# 🌀 face2sketch — Phase 3: Conditional DDPM (Kaggle)

**Build DDPM/DDIM diffusion from scratch — no HuggingFace, no diffusers.**

### How it works
- Training: add noise to sketch → U-Net predicts noise (conditioned on photo)
- Inference: start from pure noise → DDIM 50-step denoising → sketch emerges
- No discriminator — pure MSE, stable training, no adversarial issues

### Settings
- T=1000, cosine schedule, base_ch=64, time_dim=256
- LR=2e-4, batch=8, 256×256
- 500 epochs (~8-10 hr on T4 x2)

In [ ]:
# @title 1. Clone Repo + Install
import os, sys, shutil
BASE = '/kaggle/working/face2sketch'
os.makedirs(BASE, exist_ok=True)
os.chdir('/kaggle/working')

!git clone https://github.com/weseegod/face2sketch.git {BASE} 2>/dev/null || true
os.chdir(BASE)
!git pull origin main

!pip install -q tqdm matplotlib pillow pyyaml einops
!mkdir -p checkpoints samples outputs

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print(f"BASE: {BASE}")
print('\n✅  Setup complete.')

# Stick to BASE for all later cells
os.environ['FACE2SKETCH_BASE'] = BASE

In [ ]:
# @title 2. Copy Data from Kaggle Dataset
import os, shutil

BASE = os.environ.get('FACE2SKETCH_BASE', '/kaggle/working/face2sketch')
os.chdir(BASE)

# Find the folder with dataset/ and test/ subdirs (Kaggle auto-unzips)
data_root = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'dataset' in dirs and 'test' in dirs:
        data_root = root
        break

if data_root:
    print(f'📂  Found data at: {data_root}')
    os.makedirs(f'{BASE}/data', exist_ok=True)
    for sub in ['dataset', 'test']:
        src = os.path.join(data_root, sub)
        dst = f'{BASE}/data/{sub}'
        if not os.path.exists(dst):
            shutil.copytree(src, dst)
        n = len(os.listdir(os.path.join(dst, 'photos')))
        print(f'    data/{sub}/  →  {n} photos')
else:
    print('❌  dataset/ and test/ folders not found')

# Verify
ds = len(os.listdir(f'{BASE}/data/dataset/photos')) if os.path.isdir(f'{BASE}/data/dataset/photos') else 0
ts = len(os.listdir(f'{BASE}/data/test/photos')) if os.path.isdir(f'{BASE}/data/test/photos') else 0
print(f'\n✅  dataset={ds//2} pairs  |  test={ts//2} pairs')

if ds == 0:
    print('⛔  No data found. Fix dataset and re-run.')
    raise SystemExit(1)

In [ ]:
# @title 3. Train — 500 epochs (~8-10 hr)
import os
assert torch.cuda.is_available(), "⚠️  Enable GPU in Notebook settings"

BASE = os.environ.get('FACE2SKETCH_BASE', '/kaggle/working/face2sketch')
os.chdir(BASE)

print("🌀  Training Phase 3 — Conditional DDPM from scratch")
print("    T=1000 | cosine schedule | DDIM 50-step sampling")
print("    LR=2e-4 | batch=8 | 256×256 | fp32")
print(f"    322 pairs  |  500 epochs\n")

!python src/train_diffusion.py \
    --device cuda \
    --epochs 500 \
    --batch-size 8 \
    --lr 2e-4 \
    --patience 0 \
    --val-every 25 \
    --name phase3_v1_

print("\n✅  Training complete → checkpoints/phase3_v1_best.pt")


In [ ]:
# @title 4. Evaluate
import os, glob

BASE = os.environ.get('FACE2SKETCH_BASE', '/kaggle/working/face2sketch')
os.chdir(BASE)

ckpt = 'checkpoints/phase3_v1_best.pt'
if not os.path.exists(ckpt):
    candidates = sorted(glob.glob('checkpoints/phase3_v1_*.pt'), reverse=True)
    for c in candidates:
        if os.path.getsize(c) > 100000:
            ckpt = c; break

if not os.path.exists(ckpt):
    print('❌  No checkpoint found.')
elif not os.path.exists('data/test/photos'):
    print('⚠️  No test data.')
else:
    print(f'📊  Loading {ckpt}...')
    print(f'    Size: {os.path.getsize(ckpt)/1e6:.0f} MB\n')

    # Show training progress (epoch/val loss)
    import torch
    ckpt_data = torch.load(ckpt, map_location='cpu', weights_only=False)
    print(f"    Epoch: {ckpt_data.get('epoch', '?')}  |  Val loss: {ckpt_data.get('val_loss', '?'):.6f}\n")

    # Quick evaluation on test set via evaluate.py
    import sys
    sys.path.insert(0, f'{BASE}/src')
    from evaluate import evaluate_diffusion
    from diffusion import DiffusionModel

    device = torch.device('cuda')
    model = DiffusionModel(T=1000, base_ch=64, time_dim=256, device=device)
    state = ckpt_data.get('model', ckpt_data)
    if any(k.startswith('module.') for k in state.keys()):
        state = {k.replace('module.', ''): v for k, v in state.items()}
    model.model.load_state_dict(state)
    model.model.eval()

    evaluate_diffusion(model, 'data/test', device, num_steps=50)
    print('\n📸  Check outputs/phase3_test_eval.png')
    print('    Row 1 = photos  |  Row 2 = generated  |  Row 3 = ground-truth')

In [ ]:
# @title 5. Save Results to Kaggle Output
import os, shutil, glob, zipfile

BASE = os.environ.get('FACE2SKETCH_BASE', '/kaggle/working/face2sketch')
OUT  = '/kaggle/working'

# ZIP checkpoints only
checkpoint_dir = f'{BASE}/checkpoints'
zip_path = f'{OUT}/checkpoints_phase3.zip'

if os.path.exists(checkpoint_dir):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(checkpoint_dir):
            for file in files:
                if not file.startswith('phase3_v1_'):
                    continue
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, OUT)
                zipf.write(file_path, arcname)

    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f'✅ checkpoints_phase3.zip created ({size_mb:.1f} MB)')
    print('📥 Just download checkpoints_phase3.zip from the Kaggle Output tab')
else:
    print('⚠️ checkpoints/ folder not found — nothing to zip')

# Also copy evaluation images to Output
for f in glob.glob(f'{BASE}/outputs/phase3_*.png'):
    shutil.copy(f, OUT)
    print(f'📸  Copied: {os.path.basename(f)}')